In [1]:
# =========================================================
# CELL 1 – Install packages (run once if needed)
# =========================================================
!pip install yfinance nltk beautifulsoup4 requests xgboost


In [2]:
# =========================================================
# CELL 2 – Import Libraries
# =========================================================
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier


In [11]:
# =========================================================
# CELL 3 – Download META Stock Data
# =========================================================
ticker = "META"

data = yf.download(ticker, start="2023-01-01", end="2024-01-01")

# Ensure columns are single level and represent the financial metric (e.g., 'Close').
if isinstance(data.columns, pd.MultiIndex):
    # Take the first level of the MultiIndex, which typically contains 'Open', 'High', 'Close', 'Adj Close', etc.
    data.columns = data.columns.get_level_values(0)

# Ensure the column index has no 'name' attribute that might interfere with column selection.
data.columns.name = None

# Select only the 'Close' column. Handle 'Adj Close' if 'Close' is not present.
if 'Close' in data.columns:
    data = data[['Close']]
elif 'Adj Close' in data.columns:
    data = data[['Adj Close']]
    data.columns = ['Close'] # Rename 'Adj Close' to 'Close' for consistency
else:
    # If neither found, something is wrong with yfinance output.
    raise KeyError("Neither 'Close' nor 'Adj Close' column found in the downloaded data.")

# Create the 'date' column from the DataFrame's index.
data['date'] = data.index.date

# Finally, ensure the DataFrame has a simple, single-level integer index (RangeIndex).
data = data.reset_index(drop=True)

print(data.head())

/tmp/ipython-input-2471362059.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2023-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed

        Close        date
0  123.874702  2023-01-03
1  126.486450  2023-01-04
2  126.059433  2023-01-05
3  129.118088  2023-01-06
4  128.571884  2023-01-09


In [19]:
# =========================================================
# CELL 4 – Fetch News from RSS Feed
# =========================================================
# Fetch news for the same period as the stock data to ensure date overlap.
# Stock data is from 2023-01-01 to 2023-12-31.
url = "https://news.google.com/rss/search?q=META+stock+after:2023-01-01+before:2024-01-01"
response = requests.get(url)

soup = BeautifulSoup(response.content, "xml")
items = soup.find_all("item")

news = []
for item in items:
    news.append({
        "title": item.title.text,
        "date": item.pubDate.text
    })

news_df = pd.DataFrame(news)
print(news_df.head())

                                               title  \
0  Ahead of Earnings, Is Meta Stock a Buy, a Sell...   
1  Meta's stock is wrapping up a record year, spu...   
2  Meta Stock Has Room to Soar. How Ads and AI Ca...   
3  Meta Stock Is a Buy Because the Company Is Doi...   
4  Meta doubled profits in September quarter as t...   

                            date  
0  Wed, 19 Jul 2023 07:00:00 GMT  
1  Mon, 18 Dec 2023 08:00:00 GMT  
2  Fri, 22 Sep 2023 07:00:00 GMT  
3  Sun, 29 Oct 2023 07:00:00 GMT  
4  Wed, 25 Oct 2023 07:00:00 GMT  


In [20]:
# =========================================================
# CELL 5 – Sentiment Analysis using VADER
# =========================================================
nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

news_df["sentiment"] = news_df["title"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

news_df["date"] = pd.to_datetime(news_df["date"]).dt.date

print(news_df.head())

                                               title        date  sentiment
0  Ahead of Earnings, Is Meta Stock a Buy, a Sell...  2023-07-19     0.4404
1  Meta's stock is wrapping up a record year, spu...  2023-12-18    -0.7269
2  Meta Stock Has Room to Soar. How Ads and AI Ca...  2023-09-22     0.3818
3  Meta Stock Is a Buy Because the Company Is Doi...  2023-10-29    -0.3892
4  Meta doubled profits in September quarter as t...  2023-10-25     0.4404


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [21]:
# =========================================================
# CELL 6 – Aggregate Daily Sentiment & Merge with Stock Data
# =========================================================
daily_sentiment = news_df.groupby("date")["sentiment"].mean().reset_index()

# 'data' already has a 'date' column in the correct format (datetime.date)
# and a simple RangeIndex from previous steps (Cell 3).
# So we can directly use 'data' for merging without creating data_temp and data_final.

final_df = pd.merge(data,
                    daily_sentiment,
                    on="date",
                    how="inner")

print(final_df.head())

        Close        date  sentiment
0  152.057831  2023-02-01   0.557400
1  213.260300  2023-04-10   0.000000
2  211.313889  2023-04-24   0.000000
3  207.947403  2023-04-26   0.498150
4  236.905136  2023-04-27   0.221267


In [22]:
# =========================================================
# CELL 7 – Create Target Variable
# =========================================================
final_df["target"] = (
    final_df["Close"].shift(-1) > final_df["Close"]
).astype(int)

final_df.dropna(inplace=True)

print(final_df.head())


        Close        date  sentiment  target
0  152.057831  2023-02-01   0.557400       1
1  213.260300  2023-04-10   0.000000       0
2  211.313889  2023-04-24   0.000000       0
3  207.947403  2023-04-26   0.498150       1
4  236.905136  2023-04-27   0.221267       1


In [23]:
# =========================================================
# CELL 8 – Train Test Split (Time Based)
# =========================================================
X = final_df[["sentiment"]]
y = final_df["target"]

split = int(0.8 * len(final_df))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 51
Test size: 13


In [24]:
# =========================================================
# CELL 9 – Logistic Regression Model
# =========================================================
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.7692307692307693


In [28]:
# =========================================================
# CELL 10 – Random Forest Model
# =========================================================
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.6923076923076923


In [29]:
# =========================================================
# CELL 11 – XGBoost Model
# =========================================================
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))


XGBoost Accuracy: 0.6923076923076923


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:46:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [30]:
# =========================================================
# CELL 12 – Final Evaluation Report
# =========================================================
print(classification_report(y_test, y_pred_xgb))


              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.80      0.80      0.80        10

    accuracy                           0.69        13
   macro avg       0.57      0.57      0.57        13
weighted avg       0.69      0.69      0.69        13



In [31]:
# =========================================================
# Classification Report for Logistic Regression
# =========================================================
from sklearn.metrics import classification_report
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.77      1.00      0.87        10

    accuracy                           0.77        13
   macro avg       0.38      0.50      0.43        13
weighted avg       0.59      0.77      0.67        13



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [32]:
# =========================================================
# Classification Report for Random Forest
# =========================================================
from sklearn.metrics import classification_report
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.80      0.80      0.80        10

    accuracy                           0.69        13
   macro avg       0.57      0.57      0.57        13
weighted avg       0.69      0.69      0.69        13



In [33]:
# =========================================================
# Classification Report for XGBoost (already printed, re-displaying for completeness)
# =========================================================
from sklearn.metrics import classification_report
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.80      0.80      0.80        10

    accuracy                           0.69        13
   macro avg       0.57      0.57      0.57        13
weighted avg       0.69      0.69      0.69        13

